# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [7]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


In [8]:
# Shared colours (colour-blind safe: no red/green pairs)
ACCENT = '#E07B39'      # the one highlighted item (orange)
GREY = '#BBBBBB'        # context

continent_colours = {
    'Africa':        '#E07B39',   # orange
    'Asia':          '#2E75B6',   # blue
    'Europe':        '#7B5EA7',   # purple
    'North America': '#F5C842',   # yellow
    'South America': '#56B4E9',   # light blue
    'Oceania':       '#8C8C8C',   # grey
}

---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I used a line chart because the question is about change over time. Continents have very different sizes, so I plotted each continent's population relative to 1970 (1970 = 100); every line starts at the same point and growth can be compared fairly. All continents are light grey except the fastest-growing one, which is orange and thicker, and I labelled the lines directly at their ends, so no legend is needed.

In [9]:
# Q1
# Population of each continent in every year, then indexed to 1970 (1970 = 100)
cont = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
base = cont[cont['Year'] == 1970].set_index('Continent')['Population']
cont['Index'] = cont['Population'] / cont['Continent'].map(base) * 100

# Fastest growing continent = highest index in 2022
last = cont[cont['Year'] == 2022].sort_values('Index', ascending=False)
fast_name = last.iloc[0]['Continent']
fast_index = last.iloc[0]['Index']

# Grey for everyone, accent for the fastest
colour_map = {c: (ACCENT if c == fast_name else GREY) for c in cont['Continent'].unique()}

fig = px.line(
    cont, x='Year', y='Index', color='Continent',
    color_discrete_map=colour_map,
    template='plotly_white', height=500, width=900
)

fig.update_traces(line=dict(width=1.5))
fig.update_traces(line=dict(width=4), selector=dict(name=fast_name))

# Direct labels at the end of each line (no legend)
for _, r in last.iterrows():
    is_fast = r['Continent'] == fast_name
    fig.add_annotation(
        x=2022, y=r['Index'], text=f"<b>{r['Continent']}</b>" if is_fast else r['Continent'],
        showarrow=False, xanchor='left', xshift=8,
        font=dict(size=13 if is_fast else 11, color=ACCENT if is_fast else '#777777')
    )

fig.update_layout(
    title=f"{fast_name} grew fastest: its population is now {fast_index / 100:.1f}x its 1970 size",
    showlegend=False,
    xaxis_title=None,
    yaxis_title='Population (1970 = 100)',
    margin=dict(l=60, r=130, t=80, b=40),
    font=dict(family='Arial', size=12)
)
fig.update_xaxes(showgrid=False, tickvals=sorted(cont['Year'].unique()))
fig.update_yaxes(gridcolor='#EEEEEE')

fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A sorted horizontal bar chart is the clearest way to rank countries, and the long country names fit well on the y-axis. The bars are sorted from largest to smallest, the African countries are orange and all others grey, so the number of African countries can be seen at a glance. The values are written on the bars, so I removed the x-axis, the gridlines and the legend.

In [10]:
# Q2
df['Added'] = df['2022 Population'] - df['1970 Population']
df['Added (millions)'] = df['Added'] / 1e6

top15 = df.nlargest(15, 'Added').sort_values('Added')       # ascending, so the largest is at the top
n_africa = (top15['Continent'] == 'Africa').sum()
top15['Group'] = np.where(top15['Continent'] == 'Africa', 'Africa', 'Other continents')

fig = px.bar(
    top15, x='Added (millions)', y='Country', orientation='h',
    color='Group', color_discrete_map={'Africa': ACCENT, 'Other continents': GREY},
    text='Added (millions)',
    template='plotly_white', height=550, width=900
)

fig.update_traces(texttemplate='%{text:.0f}M', textposition='outside', cliponaxis=False)

fig.update_layout(
    title=f"{n_africa} of the 15 countries that added the most people since 1970 are in Africa (orange)",
    showlegend=False,
    xaxis=dict(visible=False),
    yaxis_title=None,
    margin=dict(l=20, r=60, t=80, b=20),
    font=dict(family='Arial', size=12)
)

fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A scatter plot shows the relationship between two numeric variables. Area and population range over several orders of magnitude, so both axes use a log scale; otherwise a few giant countries would squash the rest. Colour shows the continent, so differences between continents are visible. The dashed line is the overall trend, and the countries furthest from it are labelled, because those are the ones that break the pattern.

In [11]:
# Q3
# Overall trend on a log-log scale
df['log_area'] = np.log10(df['Area'])
df['log_pop'] = np.log10(df['2022 Population'])
slope, intercept = np.polyfit(df['log_area'], df['log_pop'], 1)
df['Residual'] = df['log_pop'] - (slope * df['log_area'] + intercept)
corr = df['log_area'].corr(df['log_pop'])
strength = 'strongly' if corr >= 0.7 else 'moderately' if corr >= 0.4 else 'weakly'

fig = px.scatter(
    df, x='Area', y='2022 Population', color='Continent',
    color_discrete_map=continent_colours,
    log_x=True, log_y=True, hover_name='Country',
    template='plotly_white', height=600, width=950
)

fig.update_traces(marker=dict(size=8, opacity=0.8))

# Trend line
xs = np.array([df['Area'].min(), df['Area'].max()])
fig.add_scatter(x=xs, y=10 ** (slope * np.log10(xs) + intercept),
                mode='lines', line=dict(color='#888888', dash='dash'),
                showlegend=False, hoverinfo='skip')

# Label the countries furthest from the trend (annotations on log axes use log10 values)
outliers = df.reindex(df['Residual'].abs().sort_values(ascending=False).index).head(6)
for _, r in outliers.iterrows():
    fig.add_annotation(
        x=r['log_area'], y=r['log_pop'], text=r['Country'],
        showarrow=True, arrowhead=0, arrowcolor='#999999', ax=25, ay=-25,
        font=dict(size=11, color='#333333')
    )

fig.update_layout(
    title=f"Land area and population are {strength} related (log-log r = {corr:.2f}), and a few countries break the pattern",
    xaxis_title='Land area (km², log scale)',
    yaxis_title='Population 2022 (log scale)',
    legend_title_text='',
    font=dict(family='Arial', size=12)
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(gridcolor='#EEEEEE')

fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A box plot shows the spread of density inside each continent (median, quartiles and outliers) and lets me compare continents side by side, which a bar of averages cannot do. I drew every country as a point on top of the boxes so the reader can see the individual values. Density has a few extreme values, so I used a log scale. The continents are sorted by median density, and the highest one is orange while the others are grey.

In [12]:
# Q4
order = df.groupby('Continent')['Density'].median().sort_values(ascending=False)
top_cont = order.index[0]
top_median = order.iloc[0]

# How big is the biggest gap between the least and most dense country inside one continent?
spread = df.groupby('Continent')['Density'].agg(['min', 'max'])
max_ratio = (spread['max'] / spread['min']).max()

fig = px.box(
    df, x='Continent', y='Density', color='Continent',
    color_discrete_map={c: (ACCENT if c == top_cont else GREY) for c in df['Continent'].unique()},
    category_orders={'Continent': order.index.tolist()},
    points='all', log_y=True, hover_name='Country',
    template='plotly_white', height=550, width=950
)

fig.update_layout(
    title=f"{top_cont} has the highest median density ({top_median:,.0f} people per km²), but countries within a continent differ by up to {max_ratio:,.0f}x",
    showlegend=False,
    xaxis_title=None,
    yaxis_title='People per km² (log scale)',
    font=dict(family='Arial', size=12)
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(gridcolor='#EEEEEE')

fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

The question asks where, so I used a choropleth map. The percentage change can be positive or negative, so I used a diverging colour scale centred at 0: blue for growth, red for shrinkage and white for no change. Blue and red are safe for colour-blind readers, and I did not use green. A few countries grew by much more than 60%, so I capped the colour scale at plus and minus 60% to keep the differences between most countries visible.

In [13]:
# Q5
df['Change 2000-2022 (%)'] = (df['2022 Population'] / df['2000 Population'] - 1) * 100

n_shrank = (df['Change 2000-2022 (%)'] < 0).sum()
n_grew = (df['Change 2000-2022 (%)'] > 0).sum()
shrink_by_cont = df[df['Change 2000-2022 (%)'] < 0].groupby('Continent').size()
top_shrink = shrink_by_cont.idxmax()

fig = px.choropleth(
    df, locations='CCA3', locationmode='ISO-3',
    color='Change 2000-2022 (%)',
    color_continuous_scale='RdBu', color_continuous_midpoint=0,
    range_color=[-60, 60],
    hover_name='Country', hover_data={'CCA3': False, 'Change 2000-2022 (%)': ':.1f'},
    template='plotly_white', height=550, width=1000
)

fig.update_geos(showframe=False, showcoastlines=False, projection_type='natural earth')

fig.update_layout(
    title=f"{n_shrank} countries lost people between 2000 and 2022 while {n_grew} grew; most shrinking countries are in {top_shrink}",
    coloraxis_colorbar=dict(title='% change<br>2000-2022', thickness=15, len=0.6),
    margin=dict(l=0, r=0, t=80, b=0),
    font=dict(family='Arial', size=12)
)

fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A treemap shows a whole split into two levels in one figure: the tile area is the 2022 population, the big frames are the continents, and the tiles inside are the countries. Colour shows the continent, so the reader sees which tiles belong together; the continent frames themselves are grey. The labels show the share of the world total.

In [14]:
# Q6
total = df['2022 Population'].sum()
cont_pop = df.groupby('Continent')['2022 Population'].sum().sort_values(ascending=False)
top_cont = cont_pop.index[0]
top_share = cont_pop.iloc[0] / total
two_largest = df.nlargest(2, '2022 Population')['Country'].tolist()

fig = px.treemap(
    df, path=['Continent', 'Country'], values='2022 Population',
    color='Continent', color_discrete_map=continent_colours,
    template='plotly_white', height=700, width=1100
)

fig.update_traces(
    textinfo='label+percent root',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} people<extra></extra>'
)

# Grey out the continent frames, keep colours on the country tiles
fig.data[0].marker.colors = [
    c if c in continent_colours.values() else '#DDDDDD'
    for c in fig.data[0].marker.colors
]

fig.update_layout(
    title=f"{top_cont} holds {top_share:.0%} of the world's {total / 1e9:.1f} billion people; {two_largest[0]} and {two_largest[1]} are the largest countries",
    margin=dict(l=10, r=10, t=80, b=10),
    font=dict(family='Arial', size=12)
)

fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A 100% stacked bar chart shows how the shares of a whole change from one year to the next. Africa is at the bottom, on the baseline, so its share is the easiest to compare across years, and it is the only coloured group; Asia, Europe and the rest are shades of grey. The years are treated as categories because they are not evenly spaced (1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022). The percentages are written inside the bars, so no y-axis is needed.

In [15]:
# Q7
df_long['Group'] = np.where(df_long['Continent'].isin(['Africa', 'Asia', 'Europe']),
                            df_long['Continent'], 'Rest of the world')

shares = df_long.groupby(['Year', 'Group'], as_index=False)['Population'].sum()
shares['Share'] = shares['Population'] / shares.groupby('Year')['Population'].transform('sum') * 100
shares['Label'] = shares['Share'].round(0).astype(int).astype(str) + '%'

# Biggest gainer and biggest loser among the three continents
wide = shares.pivot(index='Year', columns='Group', values='Share')
change = (wide.loc[2022] - wide.loc[1970])[['Africa', 'Asia', 'Europe']]
rise, fall = change.idxmax(), change.idxmin()

shares['Year'] = shares['Year'].astype(str)     # categorical axis: years are not evenly spaced

fig = px.bar(
    shares, x='Year', y='Share', color='Group', text='Label',
    color_discrete_map={'Africa': ACCENT, 'Asia': '#8C8C8C', 'Europe': '#BDBDBD', 'Rest of the world': '#E6E6E6'},
    category_orders={'Group': ['Africa', 'Asia', 'Europe', 'Rest of the world']},
    template='plotly_white', height=550, width=950
)

fig.update_traces(textposition='inside', textfont=dict(color='#222222', size=11),
                  marker_line_color='white', marker_line_width=1)

fig.update_layout(
    title=(f"{rise} gained the most world share ({wide.loc[1970, rise]:.0f}% to {wide.loc[2022, rise]:.0f}%), "
           f"while {fall} lost the most ({wide.loc[1970, fall]:.0f}% to {wide.loc[2022, fall]:.0f}%)"),
    xaxis_title=None,
    yaxis=dict(visible=False),
    legend_title_text='',
    bargap=0.2,
    font=dict(family='Arial', size=12)
)

fig.show()